# Experiment 2 Testing

Development-only GPU smoke for the online reference-policy Experiment 2 prototype. The source is pinned to commit `1e5dcf53973a8f876a72529e8e2d67ae79b0c21a`. This run uses tiny explicit networks, one DMC task and a short schedule. It checks accelerator visibility, source tests, online updates, matched interventions, probes, artifact integrity and checkpoint restore. It does **not** estimate the scientific hypothesis or launch the production protocol.

The Kaggle notebook must request **GPU T4 x2** and internet access. The first executable cell asserts both visible GPUs and fails if the allocation differs.


In [ ]:
import os
import subprocess
import sys
from importlib import metadata
from pathlib import Path

SOURCE_SHA = "1e5dcf53973a8f876a72529e8e2d67ae79b0c21a"
SOURCE_BRANCH = "codex/exp2-online-prototype-20261107"
REPO = Path("/kaggle/working/scaling-drl-research")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", "--branch", SOURCE_BRANCH, "https://github.com/kaves29/scaling-drl-research.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_SHA], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == SOURCE_SHA
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "main/requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]==0.4.34"], check=True)
assert metadata.version("numpy").startswith("1.26."), f"numpy drift: {metadata.version('numpy')} (numpy 1.26.x required for tensorflow-probability 0.24)"
assert metadata.version("jax") == "0.4.34", f"jax version drift: {metadata.version('jax')}"
assert metadata.version("flax") == "0.8.4", f"flax version drift: {metadata.version('flax')}"
print("Pinned source and dependencies installed:", SOURCE_SHA)


In [ ]:
# Set rendering and JAX selection before importing either library.
os.environ.update({
    "JAX_PLATFORMS": "cuda", "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl",
    "EGL_PLATFORM": "surfaceless", "MUJOCO_EGL_DEVICE_ID": "0", "WANDB_MODE": "disabled",
    "XDG_CACHE_HOME": "/kaggle/working/.cache", "MPLCONFIGDIR": "/kaggle/working/.mpl",
    "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "PYTHONDONTWRITEBYTECODE": "1",
})
subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], check=True)
import jax
names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).splitlines()
assert len(names) == 2 and all("T4" in name for name in names), f"Expected 2 T4 GPUs, got {names}"
assert jax.default_backend() == "gpu" and len(jax.devices("gpu")) == 2, jax.devices()
print("GPU backend:", jax.devices("gpu"))


In [ ]:
# Run the source-level shadow/actor contracts on the allocated GPU.
import unittest
sys.path.insert(0, str(REPO / "main"))
suite = unittest.defaultTestLoader.loadTestsFromName("tests.test_online_exp2")
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful() and result.testsRun >= 6, f"Source contracts failed: {result}"


In [ ]:
# Run the actual online driver on DMC, with small development-only sizes.
import json
import tempfile
import numpy as np
from experiments.exp1 import compose_config
from experiments.online_exp2 import OnlineExp2

cfg = compose_config(str(REPO / "main/configs"), "base_online_exp2", [
    "online_exp2.environments=[dog-run]", "online_exp2.training_seeds=[1]",
    "online_exp2.reference_critic={blocks:1,width:8}",
    "online_exp2.training_interactions=2", "online_exp2.checkpoint_steps=[2]",
    "online_exp2.probe_checkpoints=[2]", "online_exp2.evaluation_episodes=1",
    "online_exp2.evaluation_mode=deterministic", "online_exp2.intervention_updates=1",
    "online_exp2.analysis_aggregation=mean_checkpoint_means",
    "online_exp2.resource_cap_gpu_hours=0",
    "online_exp2.dev_architectures=[[small,1,8],[large,1,8]]",
    "buffer.min_length=1", "buffer.sample_batch_size=4", "buffer.max_length=32",
    "probe.rounds=1", "probe.steps=2", "probe.pool_size=4", "probe.batch_size=4",
    "probe.eval_chunk=4", "logging_per_interaction_step=100",
    "evaluation_per_interaction_step=100", "num_eval_episodes=1",
    "actor_grad_cosine_every=100",
])
output = Path(tempfile.mkdtemp(prefix="exp2_gpu_smoke_", dir="/kaggle/working"))
run = OnlineExp2(cfg, str(output))
run.run()
assert run.trainer.interaction_step == 2
assert run.shadows["small"].updates == run.shadows["large"].updates == run.trainer.update_step
print("Output:", output)


In [ ]:
# Validate every required output, then restore the committed checkpoint.
import hashlib
from experiments.exp12.state import latest_state_dir

rows = [json.loads(line) for line in (output / "online_exp2.jsonl").read_text().splitlines()]
assert {"update", "probe", "evaluation", "intervention"}.issubset({row["type"] for row in rows})
probes = [row for row in rows if row["type"] == "probe" and row["check"] == 2]
assert {row["architecture"] for row in probes} == {"small", "large"}
for row in rows:
    if "artifact" in row:
        data = Path(row["artifact"]).read_bytes()
        assert hashlib.sha256(data).hexdigest() == row["sha256"]
for row in probes:
    assert np.isfinite(row["plasticity_loss_iqm"])
    with np.load(row["artifact"], allow_pickle=False) as arrays:
        assert any(key.startswith("fresh_") for key in arrays.files)
        assert any(key.startswith("current_") for key in arrays.files)
analysis = json.loads((output / "analysis.json").read_text())
assert np.isfinite(analysis["mean_checkpoint_means"])
state = latest_state_dir(output / "state")
assert state is not None
restored = OnlineExp2(cfg, str(output))
restored.restore(state)
assert restored.trainer.interaction_step == 2
assert len(restored.rows) == len(rows)
assert restored.shadows["small"].updates == run.shadows["small"].updates
restored.trainer.close()
print("GPU smoke PASS; source:", SOURCE_SHA, "output:", output)
print("Run-level development contrast:", analysis)
